# dko-3dgs — Pipeline rápido con LingBot-Map (IMG_0884)

**Flujo autónomo**: sube el video a Drive y ejecuta todo (`Entorno de ejecución → Ejecutar todas`).
El resto pasa solo: poses con [LingBot-Map](https://github.com/Robbyant/lingbot-map)
(feed-forward, minutos en vez de horas de SfM) → modelo COLMAP → 3DGS.

**Todo el progreso se guarda en Drive** (`<carpeta>/lingbot884/`): cada etapa deja
checkpoint y al re-ejecutar el notebook se salta lo ya hecho.

Requisitos: runtime GPU grande + High-RAM · el video `IMG_0884.MOV` en cualquier
carpeta de tu MyDrive.

⚠️ Piloto: las poses feed-forward de LingBot no pasan por bundle adjustment —
la calidad final se compara contra el modelo clásico (resultado_1M) para decidir
si este es el pipeline definitivo de las tiendas.

In [ ]:
# === 0. Drive + workspace persistente ===
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import glob, os
from pathlib import Path

vids = (glob.glob('/content/drive/MyDrive/**/IMG_0884.MOV', recursive=True)
        + glob.glob('/content/drive/MyDrive/IMG_0884.MOV'))
assert vids, 'No encontré IMG_0884.MOV en tu Drive'
VIDEO_DRIVE = vids[0]
WORK = Path(os.path.dirname(VIDEO_DRIVE)) / 'lingbot884'
CKPT = WORK / 'ckpt'
CKPT.mkdir(parents=True, exist_ok=True)
done = lambda t: (CKPT / t).exists()
mark = lambda t: (CKPT / t).touch()
print('video:', VIDEO_DRIVE)
print('workspace en Drive:', WORK)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader && free -h | head -2 | tail -1

In [ ]:
# === 1. Instalar LingBot-Map + pesos + gaussian-splatting ===
import os
if not os.path.exists('/content/lingbot-map'):
    !git clone --quiet https://github.com/Robbyant/lingbot-map /content/lingbot-map
    !pip install -q -e /content/lingbot-map
    !pip install -q huggingface_hub

from huggingface_hub import hf_hub_download, list_repo_files
files = list_repo_files('robbyant/lingbot-map')
ck = next((f for f in files if 'long' in f and f.endswith(('.pt', '.pth', '.safetensors'))),
          next(f for f in files if f.endswith(('.pt', '.pth', '.safetensors'))))
MODEL_PT = hf_hub_download('robbyant/lingbot-map', ck)
print('pesos:', MODEL_PT)

if not os.path.exists('/content/gaussian-splatting'):
    !git clone --quiet --recursive https://github.com/graphdeco-inria/gaussian-splatting /content/gaussian-splatting
    !pip install -q plyfile /content/gaussian-splatting/submodules/diff-gaussian-rasterization \
        /content/gaussian-splatting/submodules/simple-knn /content/gaussian-splatting/submodules/fused-ssim
    !sed -i 's/torch.load(checkpoint)$/torch.load(checkpoint, weights_only=False)/' /content/gaussian-splatting/train.py
!pip install -q pycolmap opencv-python-headless

In [ ]:
# === 2. Video a disco local + LingBot-Map (poses en ~30-60 min) ===
import shutil, subprocess

if not os.path.exists('/content/IMG_0884.MOV'):
    shutil.copy(VIDEO_DRIVE, '/content/IMG_0884.MOV')

PRED_DRIVE = WORK / 'predicciones'
if not done('2_lingbot'):
    !cd /content/lingbot-map && python demo_render/batch_demo.py \
        --video_path /content/IMG_0884.MOV \
        --output_folder /content/lb_out \
        --model_path "$MODEL_PT" \
        --config demo_render/config/indoor.yaml \
        --mode windowed --window_size 128 --overlap_keyframes 8 --keyframe_interval 10 \
        --save_predictions
    # respaldar las predicciones a Drive (checkpoint)
    PRED_DRIVE.mkdir(exist_ok=True)
    !cp -r /content/lb_out/*.npz /content/lb_out/*.yaml "$PRED_DRIVE"/ 2>/dev/null || cp -r /content/lb_out/* "$PRED_DRIVE"/
    mark('2_lingbot')
    print('predicciones respaldadas en Drive')
else:
    print('[ckpt] LingBot ya corrió — copiando predicciones desde Drive')
    !mkdir -p /content/lb_out && cp -r "$PRED_DRIVE"/* /content/lb_out/
!ls -la /content/lb_out | head -20

In [ ]:
# === 3. Inspección del NPZ (el esquema no está documentado — lo detectamos) ===
import numpy as np, glob as g
npzs = sorted(g.glob('/content/lb_out/**/*.npz', recursive=True))
assert npzs, 'no hay NPZ — revisa que la celda 2 haya usado --save_predictions'
d = np.load(npzs[0], allow_pickle=True)
print('archivos npz:', len(npzs))
for k in d.files:
    a = d[k]
    print(f'  {k}: shape={getattr(a, "shape", "?")} dtype={getattr(a, "dtype", "?")}')
# ⚠️ REVISA este print: necesitamos identificar poses (Nx4x4 o Nx3x4),
# intrínsecos (3x3 o fx fy cx cy) y puntos (Nx3 + colores).
# La celda 4 intenta detectarlos solos; si falla, pega este print en el chat.

In [ ]:
# === 4. NPZ -> modelo COLMAP (poses + puntos + intrínsecos escalados a 4K) ===
import numpy as np, pycolmap, glob as g
from pathlib import Path

def find_key(d, cands, ndim_ok):
    for k in d.files:
        kl = k.lower()
        if any(c in kl for c in cands) and getattr(d[k], 'ndim', 0) in ndim_ok:
            return k
    return None

npzs = sorted(g.glob('/content/lb_out/**/*.npz', recursive=True))
poses_l, K_l, pts_l, rgb_l = [], [], [], []
for f in npzs:
    d = np.load(f, allow_pickle=True)
    kp = find_key(d, ['pose', 'extrinsic', 'cam2world', 'c2w', 'w2c'], (3,))
    kk = find_key(d, ['intrinsic', 'K'], (2, 3))
    kx = find_key(d, ['point', 'pts', 'xyz', 'world'], (2, 3, 4))
    kc = find_key(d, ['color', 'rgb'], (2, 3, 4))
    if kp is not None: poses_l.append(np.asarray(d[kp]))
    if kk is not None: K_l.append(np.asarray(d[kk]))
    if kx is not None: pts_l.append(np.asarray(d[kx]).reshape(-1, 3))
    if kc is not None: rgb_l.append(np.asarray(d[kc]).reshape(-1, 3))
poses = np.concatenate(poses_l) if poses_l else None
assert poses is not None, 'no detecté poses en el NPZ — pega el print de la celda 3 en el chat'
print('poses:', poses.shape)

# resolución interna de LingBot (~518x378) -> escalar intrínsecos a 3840x2160
K = np.asarray(K_l[0]) if K_l else None
if K is not None and K.ndim == 3: K = K[0]
W_SRC = 2 * K[0, 2] if K is not None else 518
scale = 3840 / W_SRC
fx = (K[0, 0] if K is not None else 0.68 * W_SRC) * scale
fy = (K[1, 1] if K is not None else 0.68 * W_SRC) * scale
print(f'focal escalada: fx={fx:.0f} fy={fy:.0f} (referencia SfM clásico: ~2923)')

rec = pycolmap.Reconstruction()
cam = pycolmap.Camera.create(1, pycolmap.CameraModelId.PINHOLE, (fx + fy) / 2, 3840, 2160)
rec.add_camera(cam)
n = len(poses)
step_pose = 1
for i in range(0, n, step_pose):
    T = np.eye(4); T[:poses[i].shape[0], :poses[i].shape[1]] = poses[i]
    # LingBot entrega cam2world; COLMAP quiere world2cam
    w2c = np.linalg.inv(T)
    im = pycolmap.Image(f'{i*10+1:05d}.jpg', pycolmap.Rigid3d(pycolmap.Rotation3d(w2c[:3, :3]), w2c[:3, 3]), 1, i + 1)
    rec.add_image(im)  # nombre asume keyframe_interval=10 sobre frames 1-based
print(f'{rec.num_images()} imágenes con pose')

if pts_l:
    pts = np.concatenate(pts_l)
    rgb = (np.concatenate(rgb_l) if rgb_l else np.full_like(pts, 128))
    if rgb.max() <= 1.0: rgb = rgb * 255
    sel = np.random.default_rng(0).choice(len(pts), min(3_000_000, len(pts)), replace=False)
    for j in sel:
        rec.add_point3D(pts[j], pycolmap.Track(), rgb[j].astype(np.uint8))
    print(f'{rec.num_points3D()} puntos 3D')

Path('/content/sparse_lb').mkdir(exist_ok=True)
rec.write('/content/sparse_lb')
!cp -r /content/sparse_lb "$WORK"/
print('modelo COLMAP guardado (local + Drive)')

In [ ]:
# === 5. Frames + layout 3DGS ===
import shutil
if not os.path.exists('/content/frames') or len(g.glob('/content/frames/*.jpg')) < 26000:
    !mkdir -p /content/frames
    !ffmpeg -hide_banner -loglevel error -i /content/IMG_0884.MOV -qscale:v 2 /content/frames/%05d.jpg

rec = pycolmap.Reconstruction('/content/sparse_lb')
names = sorted(rec.image(i).name for i in rec.reg_image_ids())
data = Path('/content/data_lb')
(data / 'images').mkdir(parents=True, exist_ok=True)
(data / 'sparse' / '0').mkdir(parents=True, exist_ok=True)
for nme in names:
    if not (data / 'images' / nme).exists():
        os.link(f'/content/frames/{nme}', data / 'images' / nme)
for f in Path('/content/sparse_lb').iterdir():
    shutil.copy(f, data / 'sparse' / '0' / f.name)
print(len(names), 'imágenes listas en', data)

In [ ]:
# === 6. Entrenamiento 3DGS + respaldo a Drive cada 30 min ===
OUT = '/content/output/dko3d_lingbot'
DEST = str(WORK / 'resultado')
os.makedirs(DEST, exist_ok=True)
os.environ['DEST_LB'] = DEST
!nohup bash -c 'while true; do cp -u /content/output/dko3d_lingbot/chkpnt*.pth "$DEST_LB/" 2>/dev/null; sleep 1800; done' >/dev/null 2>&1 &

%cd /content/gaussian-splatting
!python train.py -s /content/data_lb -m "$OUT" \
    -r 2 --data_device cpu \
    --iterations 100000 --save_iterations 50000 100000 --test_iterations -1 \
    --checkpoint_iterations 30000 60000 90000 \
    --position_lr_max_steps 100000 --densify_until_iter 30000

!cp -rv "$OUT"/point_cloud "$DEST_LB"/
!cp -v "$OUT"/cameras.json "$OUT"/cfg_args "$OUT"/chkpnt*.pth "$DEST_LB"/ 2>/dev/null
print('✅ modelo final en Drive:', DEST)

In [ ]:
# === 7. Veredicto: render vs foto real ===
%cd /content/gaussian-splatting
!timeout 600 python render.py -m /content/output/dko3d_lingbot --iteration 100000 --skip_test
import cv2, numpy as np
from pathlib import Path
R = Path('/content/output/dko3d_lingbot/train/ours_100000')
if (R / 'renders').exists():
    for nme in sorted(p.name for p in (R / 'renders').iterdir())[::800][:3]:
        r, gt = cv2.imread(str(R / 'renders' / nme)), cv2.imread(str(R / 'gt' / nme))
        cv2.imwrite(f'/content/cmp_{nme}', np.vstack([gt, r]))
        print(f'{nme}: L1 = {np.abs(r.astype(float) - gt.astype(float)).mean() / 255:.4f} (bueno <0.05)')
    !cp /content/cmp_*.png "$WORK"/ 2>/dev/null
    print('comparaciones guardadas en Drive')